# 1 Intro into torch

## 1.1 Tensors

In [ ]:
import torch
import numpy as np

Let's start with the basics. We create some dummy dataset: two observations $n$, every observation has three features $m$. We organize that as a dataset with dimensions $(n,m)$, so that is $(2,3)$

In [ ]:
data = [[1, 2, 3], [10, 20, 30]]

Our datatype here is `List[int]`, and PyTorch uses a `torch.Tensor` datatype.

In [ ]:
X = torch.tensor(data)
type(X)

We can retrieve the shape

In [ ]:
X.shape

And the type of the data inside the tensor:

In [ ]:
X.dtype

Or the amount of observations:

In [ ]:
len(X)

We can also start with a `numpy.array`

In [ ]:
npdata = np.array(data, dtype=np.float32)

Note we changed the dataformat to `np.float32`

In [ ]:
X2 = torch.from_numpy(npdata)
X2

In [ ]:
X2.dtype

## 1.2 Usefull functions for creating tensors

We can easily create a stand in tensor, with the same shape as our data:

In [ ]:
ones = torch.ones_like(X2)
ones

Or random weights. These are uniform distributed positive numbers between 0 and 1

In [ ]:
X3 = torch.rand(2, 3)
X3

If we want normally distributed numbers, we need to specify mean and standard deviation:

In [ ]:
X4 = torch.normal(mean=0.0, std=0.1, size=(2, 3))
X4

If your laptop or server has a GPU, PyTorch can run the calculations on the GPU. You can check if the GPU can be found by PyTorch with:

In [ ]:
torch.cuda.is_available()

And you can set the tensor to the GPU device with `.to()`. Default is `"cpu"`

In [ ]:
if torch.cuda.is_available():
    tensor = X3.to("cuda")
else:
    print("cuda not found")
X3.device

For people with a macbook with an `mps` backend, there is mps acceleration available.

In [ ]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = "cpu"
print(f"Using device {device}")
tensor = X3.to(device)
tensor

Please note that using accelaration with cuda or mps is not always faster!
Reasons why this can be slower are:
- Memory transer: data needs to be transfered from cpu to gpu. This can be a bottleneck.
- parallel processing limits: some architectures (especially the RNNs we will learn about in lesson 3) cant be parallelized. 
- synchronisation overhead: running things in parallel also takes some overhead to synchronise the calculations, like waiting things to finish, merging them back together, etc.

This will especially be true for the simplere models and datasets we are using in the contexts of our lessons.

Other usefull tricks are to create an array of ones. Can you figure out how to create an array of zeros for yourself?

In [ ]:
ones = torch.ones(1, 10)
ones

Tensors can be concatenated. We need to specify the dimension along which the concatenation is done:

In [ ]:
torch.cat([ones, ones, ones], dim=0)

## 1.3 Manipulation of tensors

The basis of most machine learning functions is the linear function. We can easily scale this by using matrix multiplication. Let's say we start with some random data, 32 observations with 10 features.

In [ ]:
X = torch.rand(32, 10)

Now, if we want a linear map that transforms these 10 features into 2 dimensions, we can do that with a set of weights with dimensions $(10,2)$

In [ ]:
W = torch.rand(10, 2)

In [ ]:
yhat = X @ W
yhat.shape

Equivalent is this syntax:

In [ ]:
yhat = torch.matmul(X, W)
yhat.shape

Torch will scale this up if you have more dimensions:

In [ ]:
X = torch.rand(32, 10, 16)
W = torch.rand(16, 2)
yhat = X @ W
yhat.shape

And finally, we can aggregate the tensor along the two features by taking the mean over the last dimension.

In [ ]:
aggregate = yhat.mean(dim=-1)
aggregate.shape

Try for yourself to calculate the sum

## 1.4 GPU or CPU

Tensors live in the CPU or GPU:

In [ ]:
X.device

You can check if you have a GPU available:

In [ ]:
torch.cuda.is_available()

Or a mac with M1

In [ ]:
torch.backends.mps.is_available()

And move a tensor to the GPU for faster computing, if available

In [ ]:
if torch.cuda.is_available():
    X_ = X.to("cuda")
elif torch.backends.mps.is_available():
    X_ = X.to("mps")
else:
    X_ = X.to("cpu")
X_.device

## 1.5 Reshape or View

Often, you will need to reshape a tensor:

In [ ]:
X = torch.rand(32, 28, 28, 1)
X_view = X.view(32, 28 * 28)
X_reshape = X.reshape(32, 28 * 28)
X.shape, X_view.shape, X_reshape.shape

The difference between `view` and `reshape` is: `view` operates as a view on the original tensor. If the underlying data is changed, the view will change too.

No data movement occurs when creating a view, view tensor just changes the way it interprets the same data.

In [ ]:
X = torch.Tensor([0, 0])
X_view = X.view(1, 2)
X.storage().data_ptr() == X_view.storage().data_ptr()

In [ ]:
X[0] = 1
X_view

`view` can throw an error if the required view is not contiguous (does not share the same memory block)

> A tensor whose values are laid out in the storage starting from the rightmost dimension onward (that is, moving along rows for a 2D tensor) is defined as contiguous. Contiguous tensors are convenient because we can visit them efficiently in order without jumping around in the storage (improving data locality improves performance because of the way memory access works on modern CPUs). This advantage of course depends on the way algorithms visit.

You could call `.contiugous()` on a `view`, but `.reshape()` does that behind the scenes.

## 1.6 Permute

Sometimes you might want to reshuffle the order of a tensor.

For example, let's say we load an batch of 32 images, where every image has a size of 28x28 pixels, and has 3 channels (RGB color)

In [ ]:
X = torch.rand(32, 28, 28, 3)

It is the case that there are different conventions for manipulating tensors in image recognition models. Some models have a channel-last convention, like I used above, but some (like [pytorch](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)) use a channel first convention, which would be (batch, channel, height, width).

You would want to swap the 4th dimension to the 2nd, or if you start from zero:

In [ ]:
channel_first = X.permute(0, 3, 1, 2)
channel_first.shape

## 1.7 Broadcasting

Broadcasting is something you might know from `numpy`, but it is also used by `tensorflow`, `jax` and `torch`. 

Broadcasting allows to extend a dimension, without the need to do so explicitly. The rules for broadcasting are simple:

- two dimesions are equal
- one of the dimensions is 1
- one of the dimensions does not exist

but lets show an example

In [ ]:
a = torch.ones(2, 2)
b = torch.ones(2, 2)
a, b, a + b

This is straigh forward. But what would happen in this case:

In [ ]:
a = torch.ones(1, 2)
b = torch.ones(2, 2)

`b` is a 2x2 grid, and has four numbers. If we want to add `a`, we have only two numbers! Now, you could start stacking the `a` tensor to get matching dimensions. But you dont have to!

In [ ]:
a + b

See what happened here? 

`a` is magically broadcasted over the first dimension. And what would you guess would happen in this case:

In [ ]:
a = torch.ones(1, 5, 1, 4)
b = torch.ones(3, 1, 3, 1)

First, predict the output shape, then check it for yourself.

And, what would you think happens here; do you think this gives an error, or do you think it broadcasts?

In [ ]:
a = torch.ones(5, 1, 4)
b = torch.ones(3, 1, 3, 1)

In [ ]:
(a + b).shape

## 1.8 Shape and meaning

You now know how to make tensors, reshape them, permute them and broadcast them. What you do *not*
have yet is a way to say out loud what a given tensor **means** — and it turns out that tensors live
on two levels at once. Almost every confusion later in this course comes from mixing them up, so we
will separate them here.

**The formal level: a tensor is a rank and a shape. That is all.**

`(200, 3)` is a 2D tensor with 200 rows of 3 numbers. `torch` knows nothing else about it, and every
operation you will ever call sees only this.

**The semantic level: what the axes stand for.** That same `(200, 3)` could be 200 timesteps of a
3-sensor signal, or a batch of 200 items each 3 numbers long, or one document chunked into 200
pieces, or 200 RGB pixels, or a 200x3 greyscale image.

Those are *the same tensor*. Nothing in the object distinguishes them. The meaning lives in your
head, in the variable name, and — if you are careful — in a type annotation.

### Why bother: two different questions

| Question | Level | Who answers it |
|---|---|---|
| *Will this run?* | formal | the shape rules, mechanically |
| *Does the result mean anything?* | semantic | only you |

Here is the classic case of answering the first question with the second one.

> **Can you use a CNN on a timeseries?**

Most people say no: a CNN is an image model, and a timeseries is not an image. That is a *semantic*
answer to a *formal* question, and it is wrong. Let's just try it.

In [ ]:
import torch
from torch import nn

# 32 series, 200 timesteps, 3 sensors -- (B, S, D)
tensor3d = torch.rand(32, 200, 3)

# Conv1d wants (B, C, L): channels second. Our sensors are the channels, time is the length.
conv1d = nn.Conv1d(in_channels=3, out_channels=16, kernel_size=5)
conv1d(tensor3d.permute(0, 2, 1)).shape

In [ ]:
# Conv2d wants (B, C, H, W). Add a channel axis and the series *is* a one-channel image:
# height = time, width = sensors.
conv2d = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3)

tensor4d = tensor3d.unsqueeze(1)  # add channel axis

conv2d(tensor4d).shape

Both run. No complaints, no warnings, and both are real techniques that people actually use.

"CNN = image model" is a habit of association, not a constraint. The formal constraint is only
*tensor of rank 3, channels second* or *tensor of rank 4, channels second* — and a timeseries can be arranged into
either with one `permute` or one `unsqueeze`, both of which you learned in §1.5 and §1.6.

Which frees you up to ask the question that actually matters: **does convolving along this axis
exploit real structure?** A convolution assumes locality (nearby positions are related) and
translation invariance (the same pattern means the same thing wherever it occurs). Time has both,
which is why 1d CNNs on signals work well. The *sensor* axis has neither — sensor 1 and sensor 2 are
not "near" each other in any meaningful sense — which is a good argument for making sensors the
channel axis rather than a spatial one.

That is an arguable, empirical question, and it is a much better use of your attention than
"is it allowed".

### The same permissiveness, the other way round

The formal level being loose is what let the CNN eat a timeseries. The exact same property will
also happily hide a mistake.

Here is a `Linear` layer that maps 384 features to 2 classes — a head, turning a representation into
a prediction.

In [ ]:
layer = torch.nn.Linear(384, 2)

X = torch.rand(32, 384)  # 32 items, 384 features each
layer(X).shape

32 items in, 32 predictions out. Now feed it 32 documents that have each been split into 7 chunks of
384 features. First predict what happens: an error, or a shape?

In [ ]:
X = torch.rand(32, 7, 384)
layer(X).shape

No error. `nn.Linear` acts on the **last** axis and silently accepts any leading shape.

Which means you now have 224 predictions where you maybe wanted just 32 — one per *chunk* instead of one per
*document*. The model will train. The loss will go down. It is answering a different question than
the one you asked, and nothing in pytorch will ever mention it.

Same property as the CNN case; the only difference is whether you meant it. Which is precisely why
the semantic level needs writing down somewhere the computer can see.

### Contracts: putting the meaning in the code

`jaxtyping` + `beartype` let you state what the axes mean in the signature, and check it at runtime.
The annotation `Float[Tensor, "batch dim"]` is a *semantic* claim, enforced *formally*.

In [ ]:
from beartype import beartype
from jaxtyping import Float, jaxtyped
from torch import Tensor


@jaxtyped(typechecker=beartype)
def head(x: Float[Tensor, "batch dim"]) -> Float[Tensor, "batch classes"]:
    return layer(x)


head(torch.rand(32, 384)).shape

In [ ]:
# the same wrong call as above, now with a contract on it.
# (caught, so the notebook keeps running -- read the message, that is the point)
try:
    head(torch.rand(32, 7, 384))
except Exception as e:
    print(type(e).__name__)
    print(e)

The names inside the annotation are yours to choose — `"batch steps sensors"`, `"batch chunks dim"`,
whatever the tensor actually is. jaxtyping also checks that repeated names agree between arguments,
so `"batch dim"` and `"batch classes"` pin both tensors to the same batch size.

A `TypeCheckError` from a shape contract is therefore **not noise**. It is the only mechanism that
turns a semantic mistake into a formal one — the only kind a computer can catch for you. You will
see these throughout the course, especially once we start composing `vectormesh` components. Read
them as "you are on the wrong rung of the ladder", not as "something broke".

### The ladder

Since we will keep coming back to the same few shapes, they get names. `B` = batch, `C` = chunks,
`D` = feature dimension, `S` = timesteps, `T` = tokens, `K` = classes.

| Rank | Shape | Semantics | What it is |
|:---:|---|---|---|
| 1D | `(D)` | **vector** | one item, one representation |
| 2D | `(C, D)` | **chunked item** | one item, split into `C` chunks |
| 2D | `(B, D)` | **batch of vectors** | after collation, or after aggregation |
| 3D | `(B, C, D)` | **batch of chunked items** | a batch of chunked items |

Note that `(C, D)` and `(B, D)` are **formally identical** — both are just 2D tensors, and no
operation can tell them apart. The difference is entirely semantic: one document, versus a batch of
documents. Which one you are holding depends only on whether you are before or after collation.
That ambiguity is real, and the fix is to track it deliberately rather than to hope.

The full page — which operation moves between which rungs, what the common layers require formally,
and how to read a cache's `metadata.json` — is [tensor-ladder.md](tensor-ladder.md), right next to
this notebook. Read it once now; from here on the notebooks will just say "we are at `(B, C, D)`"
and expect you to know what that means.

> **Exercise.** Fix the failing `head` call in two different ways: once by changing the input, once
> by changing the annotation. Which of the two would you actually want, if the 7 were the number of
> chunks in a document? (Hint: the answer involves getting rid of the chunk axis first — that
> operation is called *aggregation*, and it is the only rank-reducing step in the whole ladder.)